# `get_synonyms()` for All APIs

Run from the **project root** so that `scripts/` is on the Python path.

This notebook demonstrates the `get_synonyms()` pipeline for every API client in `scripts/apis_pipe/`. For each API, you will see:
- The raw data returned at each fetch step (printed by the API itself)
- The final compiled synonym DataFrame

**APIs covered:**
- Standard base-class pipeline: COL, GBIF, Index Fungorum, MushroomObserver, GenBank, FishBase
- Custom `get_synonyms()` orchestration: ITIS, Tropicos, Symbiota portals

Each section queries three cases: an **accepted name**, a **known synonym**, and **"Not species"** (a control that should return an empty result).


Notice: If you see an error that says "notebook controller is DISPOSED." when using run all, you can still run the notebook by clicking each run button one at a time. 

Notice: If any changes are made to the codebase, restart the notebook to run with the most up-to-date changes.

In [1]:
import sys, os

# Ensure the project root is on sys.path when running from notebooks/apis_pipe/
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from IPython.display import display

In [2]:
import time
import pandas as pd

pd.set_option("display.max_colwidth", None)

from scripts.config import (
    COL_PORTAL,
    GBIF_PORTAL,
    GENBANK_PORTAL,
    INDEX_FUNGORUM_PORTAL,
    ITIS_PORTAL,
    MUSHROOM_OBSERVER_PORTAL,
    TROPICOS_PORTAL,
    FISHBASE_PORTAL,
    SYMBIOTA_PORTAL_BY_NAME,
)


def show(df):
    """Display a synonym DataFrame with api_link rendered as a clickable anchor."""
    if df.empty:
        display(df)
        return
    display(
        df.style.format(
            {
                "api_link": lambda url: (
                    f'<a href="{url}" target="_blank">{url}</a>'
                    if url and url not in ("U", "")
                    else url
                )
            }
        )
    )


def run_queries(queries, sleep_between_species=0.0):
    """
    Run get_synonyms() for each (portal, client, accepted_name, synonym_name) tuple.

    For each entry, queries three cases in order:
      1. The accepted name
      2. A known synonym of the accepted name
      3. "Not species" — a control that should return an empty result

    Parameters
    ----------
    queries : list of (portal, client, accepted_name, synonym_name)
    sleep_between_species : float
        Seconds to sleep between each species query. Used for rate-limited APIs.
    """
    for portal, client, accepted, synonym in queries:
        for species, name_type in [
            (accepted, "accepted"),
            (synonym, "synonym"),
            ("Not species", "not species"),
        ]:
            print(f"\n{'='*60}")
            print(f"  {portal.display_name}  ({client.__class__.__name__})")
            print(f"  Species:   {species}")
            if name_type == "synonym":
                print(f"  Name type: {name_type}  (synonym of: {accepted})")
            else:
                print(f"  Name type: {name_type}")
            print(f"{'='*60}")
            try:
                result = client.get_synonyms(species)
                print(f"Synonyms returned: {len(result)}")
                show(result)
            except Exception as e:
                print(f"ERROR: {e}")
            if sleep_between_species:
                time.sleep(sleep_between_species)

---
## How the Standard `get_synonyms()` Pipeline Works

Most API clients in this project inherit `get_synonyms()` directly from the base class (`SpeciesAPI`) without overriding it. The method orchestrates a fixed four-step pipeline:

```python
def get_synonyms(self, name: str) -> pd.DataFrame:
    name = normalize_query_string(name)

    raw_data = self._fetch_query_data(name)
    if self._is_empty(raw_data):
        return empty_synonym_table()

    synonym_data = self._fetch_synonym_data(raw_data)
    accepted_data = self._fetch_accepted_data(raw_data, synonym_data)

    synonyms = self._compile_synonyms(synonym_data) if not self._is_empty(synonym_data) else []
    accepted = self._compile_accepted(accepted_data)  if not self._is_empty(accepted_data) else []

    rows = accepted + synonyms
    return pd.DataFrame(rows) if rows else empty_synonym_table()
```

**Step 1 — `_fetch_query_data(name)`**
Makes the initial API request for the search term. Returns the raw response in the API's native format (JSON dict, list, or XML element). If nothing is found, the pipeline short-circuits and returns an empty DataFrame immediately.

**Step 2 — `_fetch_synonym_data(raw_data)`**
Uses the raw query result to fetch the list of synonyms. For most APIs this means extracting an internal ID from `raw_data` and making a second request to the synonyms endpoint.

**Step 3 — `_fetch_accepted_data(raw_data, synonym_data)`**
Fetches the full record for the accepted name — taxonomy, authorship, publication metadata, etc. Some APIs return the accepted name inline with query results (`raw_data` is reused); others require a separate request.

**Step 4 — `_compile_accepted` and `_compile_synonyms`**
Convert the raw API responses into pipeline-standard row dicts. `_compile_accepted` produces one row for the accepted name (with full taxonomy). `_compile_synonyms` produces one row per synonym. Both call `_format_row` internally to enforce the output schema. The rows are concatenated into the final DataFrame.

**What you'll see printed**
Each `get_synonyms()` call prints the raw API response after each fetch step, labelled `_fetch_query_data`, `_fetch_synonym_data`, and `_fetch_accepted_data`. This is intentional — it lets you inspect exactly what each API returns before any compilation happens.

---
## COL (Catalogue of Life)

COL is a global taxonomic checklist aggregating accepted names and synonymies across all kingdoms, maintained by a consortium of taxonomists. This client queries the ChecklistBank REST API, pinned to the COL26.5 dataset.

Uses the **standard base-class** `get_synonyms()` pipeline.

In [3]:
from scripts.apis_pipe.col import COLAPI

run_queries([(COL_PORTAL, COLAPI(), "Quercus robur", "Quercus atrosanguinea")])


  COL  (COLAPI)
  Species:   Quercus robur
  Name type: accepted
_fetch_query_data
{'offset': 0, 'limit': 10, 'total': 2, 'result': [{'id': '4R5YN', 'classification': [{'id': 'CS5HF', 'name': 'Eukaryota', 'authorship': '(Chatton, 1925) Whittaker & Margulis, 1978', 'rank': 'domain', 'status': 'accepted', 'labelHtml': 'Eukaryota (Chatton, 1925) Whittaker & Margulis, 1978'}, {'id': 'P', 'name': 'Plantae', 'rank': 'kingdom', 'status': 'accepted', 'labelHtml': 'Plantae'}, {'id': 'CMQ8S', 'name': 'Pteridobiotina', 'authorship': 'Britton & Brown', 'rank': 'subkingdom', 'status': 'accepted', 'labelHtml': 'Pteridobiotina Britton & Brown'}, {'id': 'TP', 'name': 'Tracheophyta', 'rank': 'phylum', 'status': 'accepted', 'labelHtml': 'Tracheophyta'}, {'id': 'MG', 'name': 'Magnoliopsida', 'rank': 'class', 'status': 'accepted', 'labelHtml': 'Magnoliopsida'}, {'id': '384', 'name': 'Fagales', 'authorship': 'Engl.', 'rank': 'order', 'code': 'botanical', 'status': 'accepted', 'labelHtml': 'Fagales Engl.'}

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,COL,Plantae,Tracheophyta,Magnoliopsida,Fagaceae,Fagales,Quercoideae,Quercus,robur,L.,N/A,N/A,Accepted,http://www.worldplants.de/?deeplink=Quercus-robur,https://www.catalogueoflife.org/data/taxon/4R5YN,4R5YN
1,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,atrosanguinea,hort.,N/A,N/A,Synonym,https://list.worldfloraonline.org/wfo-0000289653,https://www.catalogueoflife.org/data/taxon/SF5RP,SF5RP
2,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,belgica,Gand.,N/A,N/A,Synonym,https://list.worldfloraonline.org/wfo-0000289743,https://www.catalogueoflife.org/data/taxon/8X4Z4,8X4Z4
3,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,cupressoides,hort.,N/A,N/A,Synonym,https://list.worldfloraonline.org/wfo-0000290388,https://www.catalogueoflife.org/data/taxon/SFBKJ,SFBKJ
4,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,feminea,Mill.,N/A,N/A,Synonym,http://reflora.jbrj.gov.br/reflora/listaBrasil/FichaPublicaTaxonUC/FichaPublicaTaxonUC.do?id=FB613397,https://www.catalogueoflife.org/data/taxon/SFDVF,SFDVF
5,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,filicifolia,hort. ex A.DC.,N/A,N/A,Synonym,https://list.worldfloraonline.org/wfo-0000290788,https://www.catalogueoflife.org/data/taxon/SFDXP,SFDXP
6,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,foemida,Mill.,N/A,N/A,Synonym,https://list.worldfloraonline.org/wfo-0000290808,https://www.catalogueoflife.org/data/taxon/4R4Y4,4R4Y4
7,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,hentzei,Petz.,N/A,N/A,Synonym,https://list.worldfloraonline.org/wfo-0000291092,https://www.catalogueoflife.org/data/taxon/SFGZ6,SFGZ6
8,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,hohenackeri,Gand.,N/A,N/A,Synonym,https://list.worldfloraonline.org/wfo-0000291126,https://www.catalogueoflife.org/data/taxon/8X52M,8X52M
9,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,louettii,Dippel,N/A,N/A,Synonym,https://list.worldfloraonline.org/wfo-0000291768,https://www.catalogueoflife.org/data/taxon/SFMTB,SFMTB



  COL  (COLAPI)
  Species:   Quercus atrosanguinea
  Name type: synonym  (synonym of: Quercus robur)
_fetch_query_data
{'offset': 0, 'limit': 10, 'total': 2, 'result': [{'id': '4R4BH', 'classification': [{'id': 'CS5HF', 'name': 'Eukaryota', 'authorship': '(Chatton, 1925) Whittaker & Margulis, 1978', 'rank': 'domain', 'status': 'accepted', 'labelHtml': 'Eukaryota (Chatton, 1925) Whittaker & Margulis, 1978'}, {'id': 'P', 'name': 'Plantae', 'rank': 'kingdom', 'status': 'accepted', 'labelHtml': 'Plantae'}, {'id': 'CMQ8S', 'name': 'Pteridobiotina', 'authorship': 'Britton & Brown', 'rank': 'subkingdom', 'status': 'accepted', 'labelHtml': 'Pteridobiotina Britton & Brown'}, {'id': 'TP', 'name': 'Tracheophyta', 'rank': 'phylum', 'status': 'accepted', 'labelHtml': 'Tracheophyta'}, {'id': 'MG', 'name': 'Magnoliopsida', 'rank': 'class', 'status': 'accepted', 'labelHtml': 'Magnoliopsida'}, {'id': '384', 'name': 'Fagales', 'authorship': 'Engl.', 'rank': 'order', 'code': 'botanical', 'status': 'acce

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,COL,Plantae,Tracheophyta,Magnoliopsida,Fagaceae,Fagales,Quercoideae,Quercus,robur,,N/A,N/A,Accepted,http://www.worldplants.de/?deeplink=Quercus-robur-subsp.-robur,https://www.catalogueoflife.org/data/taxon/5KTTT,5KTTT
1,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,abbreviata,Vuk.,N/A,N/A,Synonym,,https://www.catalogueoflife.org/data/taxon/4R45Z,4R45Z
2,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,accessiva,Gand.,N/A,N/A,Synonym,,https://www.catalogueoflife.org/data/taxon/8X4Y9,8X4Y9
3,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,accomodata,Gand.,N/A,N/A,Synonym,,https://www.catalogueoflife.org/data/taxon/8X4YB,8X4YB
4,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,acutiloba,Borbás,N/A,N/A,Synonym,,https://www.catalogueoflife.org/data/taxon/4R46X,4R46X
5,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,aesculus,Boiss.,N/A,N/A,Synonym,,https://www.catalogueoflife.org/data/taxon/4R47D,4R47D
6,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,aesculus,Heuff.,N/A,N/A,Synonym,,https://www.catalogueoflife.org/data/taxon/9V354,9V354
7,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,aestivalis,Steven,N/A,N/A,Synonym,,https://www.catalogueoflife.org/data/taxon/78RQJ,78RQJ
8,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,afghanistanensis,K.Koch,N/A,N/A,Synonym,,https://www.catalogueoflife.org/data/taxon/SF45D,SF45D
9,COL,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,alligata,Gand.,N/A,N/A,Synonym,,https://www.catalogueoflife.org/data/taxon/8X4YG,8X4YG



  COL  (COLAPI)
  Species:   Not species
  Name type: not species
Synonyms returned: 0


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id


---
## GBIF (Global Biodiversity Information Facility)

GBIF aggregates occurrence records and taxonomic data contributed by institutions worldwide. This client uses the GBIF backbone taxonomy via `/species/match` to resolve a name to a usage key, then retrieves its synonym list from `/species/{id}/synonyms`.

Uses the **standard base-class** `get_synonyms()` pipeline.

In [4]:
from scripts.apis_pipe.gbif import GBIFAPI

run_queries([(GBIF_PORTAL, GBIFAPI(), "Amanita muscaria", "Agaricus muscarius")])


  GBIF  (GBIFAPI)
  Species:   Amanita muscaria
  Name type: accepted
_fetch_query_data
{'usageKey': 8168319, 'scientificName': 'Amanita muscaria (L.) Lam.', 'canonicalName': 'Amanita muscaria', 'rank': 'SPECIES', 'status': 'ACCEPTED', 'confidence': 98, 'matchType': 'EXACT', 'kingdom': 'Fungi', 'phylum': 'Basidiomycota', 'order': 'Agaricales', 'family': 'Amanitaceae', 'genus': 'Amanita', 'species': 'Amanita muscaria', 'kingdomKey': 5, 'phylumKey': 34, 'classKey': 186, 'orderKey': 1499, 'familyKey': 4171, 'genusKey': 6005964, 'speciesKey': 8168319, 'class': 'Agaricomycetes'}
_fetch_synonym_data
[{'key': 5455639, 'nubKey': 5455639, 'nameKey': 304921, 'taxonID': 'gbif:5455639', 'sourceTaxonKey': 176053019, 'kingdom': 'Fungi', 'phylum': 'Basidiomycota', 'order': 'Agaricales', 'family': 'Amanitaceae', 'genus': 'Amanita', 'species': 'Amanita muscaria', 'kingdomKey': 5, 'phylumKey': 34, 'classKey': 186, 'orderKey': 1499, 'familyKey': 4171, 'genusKey': 6005964, 'speciesKey': 8168319, 'dataset

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,GBIF,Fungi,Basidiomycota,Agaricomycetes,Amanitaceae,Agaricales,N/A,Amanita,muscaria,(L.) Lam.,(1783). Encycl. Méth. Bot. (Paris) 1(1): 111.,1783,Accepted,N/A,https://www.gbif.org/species/8168319,8168319
1,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,aureolus,Kalchbr.,(1873). Icon. Sel. Hymenomyc. Hung. (Budapest) 1: 9.,1873,Synonym,N/A,https://www.gbif.org/species/5455639,5455639
2,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,imperialis,Batsch,(1783). Elench. Fung. (Halle): 59.,1783,Synonym,N/A,https://www.gbif.org/species/5955425,5955425
3,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,muscarius,L.,(1753). Sp. Pl. 2: 1172.,1753,Synonym,N/A,https://www.gbif.org/species/5451774,5451774
4,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,nobilis,Bolton,"(1788). Hist. Fung. Halifax (Huddersfield) 2: 46, Tab. 46.",1788,Synonym,N/A,https://www.gbif.org/species/3329091,3329091
5,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,pseudoaurantiacus,Bull.,"(1792). Hist. Champ. Fr. (Paris) 2(2): 673, Pl. 122.",1792,Synonym,N/A,https://www.gbif.org/species/3329092,3329092
6,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,puellus,Batsch,(1786). Elench. Fung. (Halle): 59.,1786,Synonym,N/A,https://www.gbif.org/species/5452606,5452606
7,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,aureola,(Kalchbr.) Sacc.,(1887). Syll. Fung. (Abellini) 5: 12.,1887,Synonym,N/A,https://www.gbif.org/species/5452441,5452441
8,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,circinnata,Gray,(1821). Nat. Arr. Brit. Pl. (London) 1: 600.,1821,Synonym,N/A,https://www.gbif.org/species/5452605,5452605
9,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,formosa,Gonn. & Rabenh.,"(1869). Myc. Europ. (Dresden) 1-2: Tab. 10, Fig. 2.",1869,Synonym,N/A,https://www.gbif.org/species/5452657,5452657



  GBIF  (GBIFAPI)
  Species:   Agaricus muscarius
  Name type: synonym  (synonym of: Amanita muscaria)
_fetch_query_data
{'usageKey': 5451774, 'acceptedUsageKey': 8168319, 'scientificName': 'Agaricus muscarius L.', 'canonicalName': 'Agaricus muscarius', 'rank': 'SPECIES', 'status': 'SYNONYM', 'confidence': 99, 'matchType': 'EXACT', 'kingdom': 'Fungi', 'phylum': 'Basidiomycota', 'order': 'Agaricales', 'family': 'Amanitaceae', 'genus': 'Amanita', 'species': 'Amanita muscaria', 'kingdomKey': 5, 'phylumKey': 34, 'classKey': 186, 'orderKey': 1499, 'familyKey': 4171, 'genusKey': 6005964, 'speciesKey': 8168319, 'class': 'Agaricomycetes'}
_fetch_synonym_data
[{'key': 5455639, 'nubKey': 5455639, 'nameKey': 304921, 'taxonID': 'gbif:5455639', 'sourceTaxonKey': 176053019, 'kingdom': 'Fungi', 'phylum': 'Basidiomycota', 'order': 'Agaricales', 'family': 'Amanitaceae', 'genus': 'Amanita', 'species': 'Amanita muscaria', 'kingdomKey': 5, 'phylumKey': 34, 'classKey': 186, 'orderKey': 1499, 'familyKey': 

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,GBIF,Fungi,Basidiomycota,Agaricomycetes,Amanitaceae,Agaricales,N/A,Amanita,muscaria,(L.) Lam.,(1783). Encycl. Méth. Bot. (Paris) 1(1): 111.,1783,Accepted,N/A,https://www.gbif.org/species/8168319,8168319
1,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,aureolus,Kalchbr.,(1873). Icon. Sel. Hymenomyc. Hung. (Budapest) 1: 9.,1873,Synonym,N/A,https://www.gbif.org/species/5455639,5455639
2,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,imperialis,Batsch,(1783). Elench. Fung. (Halle): 59.,1783,Synonym,N/A,https://www.gbif.org/species/5955425,5955425
3,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,muscarius,L.,(1753). Sp. Pl. 2: 1172.,1753,Synonym,N/A,https://www.gbif.org/species/5451774,5451774
4,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,nobilis,Bolton,"(1788). Hist. Fung. Halifax (Huddersfield) 2: 46, Tab. 46.",1788,Synonym,N/A,https://www.gbif.org/species/3329091,3329091
5,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,pseudoaurantiacus,Bull.,"(1792). Hist. Champ. Fr. (Paris) 2(2): 673, Pl. 122.",1792,Synonym,N/A,https://www.gbif.org/species/3329092,3329092
6,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,puellus,Batsch,(1786). Elench. Fung. (Halle): 59.,1786,Synonym,N/A,https://www.gbif.org/species/5452606,5452606
7,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,aureola,(Kalchbr.) Sacc.,(1887). Syll. Fung. (Abellini) 5: 12.,1887,Synonym,N/A,https://www.gbif.org/species/5452441,5452441
8,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,circinnata,Gray,(1821). Nat. Arr. Brit. Pl. (London) 1: 600.,1821,Synonym,N/A,https://www.gbif.org/species/5452605,5452605
9,GBIF,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,formosa,Gonn. & Rabenh.,"(1869). Myc. Europ. (Dresden) 1-2: Tab. 10, Fig. 2.",1869,Synonym,N/A,https://www.gbif.org/species/5452657,5452657



  GBIF  (GBIFAPI)
  Species:   Not species
  Name type: not species
Synonyms returned: 0


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id


---
## Index Fungorum

Index Fungorum is the global nomenclatural database for fungi, maintained by the Royal Botanic Gardens Kew, CAB International, and Landcare Research. It exposes a legacy ASMX web service (XML over HTTP) rather than a REST/JSON API; all responses are parsed as XML. The service can be slow and may time out; requests use a 60-second timeout.

Uses the **standard base-class** `get_synonyms()` pipeline.

In [5]:
from scripts.apis_pipe.index_fungorum import IndexFungorumAPI

run_queries([(INDEX_FUNGORUM_PORTAL, IndexFungorumAPI(), "Amanita muscaria", "Agaricus muscarius")])


  Index Fungorum  (IndexFungorumAPI)
  Species:   Amanita muscaria
  Name type: accepted
_fetch_query_data
<Element 'IndexFungorum' at 0x1749dc9f0>
_fetch_synonym_data
<NewDataSet>
  <IndexFungorum>
    <NAME_x0020_OF_x0020_FUNGUS>Agaricus aureolus</NAME_x0020_OF_x0020_FUNGUS>
    <AUTHORS>Kalchbr.</AUTHORS>
    <SPECIFIC_x0020_EPITHET>aureolus</SPECIFIC_x0020_EPITHET>
    <INFRASPECIFIC_x0020_RANK>sp.</INFRASPECIFIC_x0020_RANK>
    <VOLUME>1</VOLUME>
    <PAGE>9</PAGE>
    <YEAR_x0020_OF_x0020_PUBLICATION>1873</YEAR_x0020_OF_x0020_PUBLICATION>
    <LITERATURE_x0020_LINK>3210</LITERATURE_x0020_LINK>
    <RECORD_x0020_NUMBER>298478</RECORD_x0020_NUMBER>
    <BASIONYM_x0020_RECORD_x0020_NUMBER>298478</BASIONYM_x0020_RECORD_x0020_NUMBER>
    <PROTONYM_x0020_RECORD_x0020_NUMBER>298478</PROTONYM_x0020_RECORD_x0020_NUMBER>
    <NAME_x0020_OF_x0020_FUNGUS_x0020_FUNDIC_x0020_RECORD_x0020_NUMBER>17030</NAME_x0020_OF_x0020_FUNGUS_x0020_FUNDIC_x0020_RECORD_x0020_NUMBER>
    <CURRENT_x0020_NAME>A

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,muscaria,(L.) Lam.,N/A,1783,Accepted,Saccardo's Syll. fung. V: 13; XII: 906; XIX: 49,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=161267,161267
1,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,aureolus,Kalchbr.,N/A,1873,Synonym,,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=298478,298478
2,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,imperialis,Batsch,N/A,1783,Synonym,,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=495382,495382
3,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,muscarius,L.,N/A,1753,Synonym,,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=375287,375287
4,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,nobilis,Bolton,N/A,1788,Synonym,Saccardo's Syll. fung. V: 13 | Index of Fungi 6: 1083,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=450473,450473
5,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,pseudoaurantiacus,Bull.,N/A,1792,Synonym,,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=494787,494787
6,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,puellus,Batsch,N/A,1783,Synonym,,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=372222,372222
7,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,aureola,(Kalchbr.) Sacc.,N/A,1887,Synonym,Saccardo's Syll. fung. V: 12; XII: 905; XIX: 44,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=209653,209653
8,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,circinnata,Gray,N/A,1821,Synonym,,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=486444,486444
9,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,formosa,Gonn. & Rabenh.,N/A,1869,Synonym,Saccardo's Syll. fung. V: 13; XV: 44,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=212977,212977



  Index Fungorum  (IndexFungorumAPI)
  Species:   Agaricus muscarius
  Name type: synonym  (synonym of: Amanita muscaria)
_fetch_query_data
<Element 'IndexFungorum' at 0x1749ebf60>
_fetch_synonym_data
<NewDataSet>
  <IndexFungorum>
    <NAME_x0020_OF_x0020_FUNGUS>Agaricus aureolus</NAME_x0020_OF_x0020_FUNGUS>
    <AUTHORS>Kalchbr.</AUTHORS>
    <SPECIFIC_x0020_EPITHET>aureolus</SPECIFIC_x0020_EPITHET>
    <INFRASPECIFIC_x0020_RANK>sp.</INFRASPECIFIC_x0020_RANK>
    <VOLUME>1</VOLUME>
    <PAGE>9</PAGE>
    <YEAR_x0020_OF_x0020_PUBLICATION>1873</YEAR_x0020_OF_x0020_PUBLICATION>
    <LITERATURE_x0020_LINK>3210</LITERATURE_x0020_LINK>
    <RECORD_x0020_NUMBER>298478</RECORD_x0020_NUMBER>
    <BASIONYM_x0020_RECORD_x0020_NUMBER>298478</BASIONYM_x0020_RECORD_x0020_NUMBER>
    <PROTONYM_x0020_RECORD_x0020_NUMBER>298478</PROTONYM_x0020_RECORD_x0020_NUMBER>
    <NAME_x0020_OF_x0020_FUNGUS_x0020_FUNDIC_x0020_RECORD_x0020_NUMBER>17030</NAME_x0020_OF_x0020_FUNGUS_x0020_FUNDIC_x0020_RECORD_x0020_

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,muscaria,(L.) Lam.,N/A,1783,Accepted,Saccardo's Syll. fung. V: 13; XII: 906; XIX: 49,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=161267,161267
1,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,aureolus,Kalchbr.,N/A,1873,Synonym,,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=298478,298478
2,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,imperialis,Batsch,N/A,1783,Synonym,,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=495382,495382
3,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,muscarius,L.,N/A,1753,Synonym,,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=375287,375287
4,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,nobilis,Bolton,N/A,1788,Synonym,Saccardo's Syll. fung. V: 13 | Index of Fungi 6: 1083,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=450473,450473
5,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,pseudoaurantiacus,Bull.,N/A,1792,Synonym,,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=494787,494787
6,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,puellus,Batsch,N/A,1783,Synonym,,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=372222,372222
7,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,aureola,(Kalchbr.) Sacc.,N/A,1887,Synonym,Saccardo's Syll. fung. V: 12; XII: 905; XIX: 44,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=209653,209653
8,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,circinnata,Gray,N/A,1821,Synonym,,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=486444,486444
9,Index Fungorum,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,formosa,Gonn. & Rabenh.,N/A,1869,Synonym,Saccardo's Syll. fung. V: 13; XV: 44,https://www.indexfungorum.org/Names/NamesRecord.asp?RecordID=212977,212977



  Index Fungorum  (IndexFungorumAPI)
  Species:   Not species
  Name type: not species
Synonyms returned: 0


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id


---
## Mushroom Observer

Mushroom Observer is a community-driven database of fungal observations. The JSON API embeds synonym lists directly in each name record, so synonyms require no second network request. Rather than "accepted"/"synonym" labels, Mushroom Observer uses a `deprecated` boolean flag; status is inferred from that field.

Uses the **standard base-class** `get_synonyms()` pipeline.

In [6]:
from scripts.apis_pipe.mushroomobs import MushroomObserverAPI

run_queries([(MUSHROOM_OBSERVER_PORTAL, MushroomObserverAPI(), "Amanita muscaria", "Amanita amerimuscaria")])


  Mushroom Observer  (MushroomObserverAPI)
  Species:   Amanita muscaria
  Name type: accepted
_fetch_query_data
{'version': 2.0, 'run_date': '2026-06-23T06:27:06.830Z', 'query': 'SELECT DISTINCT `names`.* FROM `names` WHERE (`names`.`id` IN (374, 5066, 16015, 28495, 41324, 59384, 373, 31199, 59289, 109028, 110413, 111780)) AND (`names`.`correct_spelling_id` IS NULL) AND (`names`.`correct_spelling_id` IS NULL) ORDER BY `names`.`id` ASC', 'number_of_records': 7, 'number_of_pages': 1, 'page_number': 1, 'results': [{'id': 373, 'type': 'name', 'name': 'Amanita muscaria', 'author': '(L.) Lam.', 'rank': 'species', 'deprecated': False, 'misspelled': False, 'citation': '<a href="https://biodiversitylibrary.org/page/716703"><cite>Encycl. Méth. Bot.</cite> (Paris) 1: 111</a> (1783)', 'created_at': '2007-01-10T05:03:59.000Z', 'updated_at': '2025-04-02T17:58:46.000Z', 'number_of_views': 7427, 'last_viewed': '2026-06-22T19:14:25.000Z', 'ok_for_export': True, 'parents': [{'name': 'Eukarya', 'rank':

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,Mushroom Observer,Fungi,Basidiomycota,Agaricomycetes,Amanitaceae,Agaricales,N/A,Amanita,muscaria,(L.) Lam.,Encycl. Méth. Bot.,1783,Accepted,N/A,https://mushroomobserver.org/names/373,373
1,Mushroom Observer,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,amerimuscaria,Tulloss & Geml nom. prov.,N/A,N/A,Synonym,N/A,https://mushroomobserver.org/names/16015,16015



  Mushroom Observer  (MushroomObserverAPI)
  Species:   Amanita amerimuscaria
  Name type: synonym  (synonym of: Amanita muscaria)
_fetch_query_data
{'version': 2.0, 'run_date': '2026-06-23T06:27:07.832Z', 'query': 'SELECT DISTINCT `names`.* FROM `names` WHERE (`names`.`id` IN (374, 5066, 16015, 28495, 41324, 59384)) AND (`names`.`correct_spelling_id` IS NULL) AND (`names`.`correct_spelling_id` IS NULL) ORDER BY `names`.`id` ASC', 'number_of_records': 6, 'number_of_pages': 1, 'page_number': 1, 'results': [{'id': 374, 'type': 'name', 'name': 'Amanita muscaria var. flavivolvata', 'author': '(Singer) Dav. T. Jenkins', 'rank': 'variety', 'deprecated': True, 'misspelled': False, 'citation': 'Bibliotheca Mycologica 57: 56 (1977) [MB#487346]', 'created_at': '2007-01-10T05:03:59.000Z', 'updated_at': '2025-05-23T22:24:35.000Z', 'number_of_views': 1024, 'last_viewed': '2025-12-17T19:19:52.000Z', 'ok_for_export': True, 'parents': [{'name': 'Eukarya', 'rank': 'domain'}, {'name': 'Fungi', 'rank': 

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,Mushroom Observer,N/A,N/A,N/A,N/A,N/A,N/A,Amanita,amerimuscaria,Tulloss & Geml nom. prov.,N/A,N/A,Synonym,N/A,https://mushroomobserver.org/names/16015,16015



  Mushroom Observer  (MushroomObserverAPI)
  Species:   Not species
  Name type: not species
_fetch_query_data
{'version': 2.0, 'run_date': '2026-06-23T06:27:08.834Z', 'errors': [{'code': 'API2::ObjectNotFoundByString', 'details': 'Name "Not species" does not exist, or someone has deleted it.', 'fatal': 'true', 'trace': "/var/web/mo/app/classes/api2/parsers/name_parser.rb:23:in 'API2::Parsers::NameParser#try_finding_by_string'\n/var/web/mo/app/classes/api2/parsers/name_parser.rb:13:in 'API2::Parsers::NameParser#find_object'\n/var/web/mo/app/classes/api2/parsers/object_base.rb:10:in 'API2::Parsers::ObjectBase#parse'\n/var/web/mo/app/classes/api2/parsers/base.rb:25:in 'API2::Parsers::Base#parse_scalar'\n/var/web/mo/app/classes/api2/parsers/base.rb:45:in 'API2::Parsers::Base#parse_array'\n/var/web/mo/app/classes/api2/core/parameters.rb:94:in 'API2::Parameters#parse_array'\n/var/web/mo/app/classes/api2/name_api.rb:23:in 'API2::NameAPI#query_params'\n/var/web/mo/app/classes/api2/model_api.

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id


---
## Tropicos

Tropicos is a botanical nomenclature database maintained by the Missouri Botanical Garden, covering vascular plants, bryophytes, algae, and fungi. **Requires a `TROPICOS_API_KEY` in your `.env` file.**

**Custom `get_synonyms()` orchestration.** Unlike the base class, which passes raw data through a fixed pipeline, Tropicos resolves the accepted `NameId` explicitly before fetching synonyms. The fetch sequence is:

1. `_fetch_query_data` — name search
2. `_fetch_accepted_list` — `/Name/{id}/AcceptedNames`, used for ID resolution only
3. `_extract_internal_accepted_id` — reads the accepted NameId (no API call)
4. `_fetch_synonym_data` — `/Name/{accepted_id}/Synonyms`
5. `_fetch_accepted_data` — `/Name/Search` by NameId *(only when the queried name is a synonym; otherwise `raw_data` is reused directly to avoid a redundant fetch)*

This two-step ID resolution is necessary because Tropicos's synonym endpoint requires the *accepted* name's ID, not the queried name's ID — and the queried name may itself be a synonym.

In [7]:
from scripts.apis_pipe.tropicos import TropicosAPI

run_queries([(TROPICOS_PORTAL, TropicosAPI(), "Quercus robur", "Quercus pedunculata")])


  Tropicos  (TropicosAPI)
  Species:   Quercus robur
  Name type: accepted
_fetch_query_data
[{'NameId': 13100004, 'ScientificName': 'Quercus robur', 'ScientificNameWithAuthors': 'Quercus robur L.', 'Family': 'Fagaceae', 'RankAbbreviation': 'sp.', 'NomenclatureStatusID': 1, 'NomenclatureStatusName': 'Legitimate', 'Symbol': '!', 'Author': 'L.', 'DisplayReference': 'Sp. Pl. 2: 996', 'DisplayDate': '1753', 'TotalRows': 2}, {'NameId': 50280607, 'ScientificName': 'Quercus robur', 'ScientificNameWithAuthors': 'Quercus robur (Ten.) A. DC.', 'Family': 'Fagaceae', 'RankAbbreviation': 'sp.', 'NomenclatureStatusName': 'No opinion', 'Author': '(Ten.) A. DC.', 'DisplayReference': 'Prodr. 16(2): 5', 'DisplayDate': '1864', 'TotalRows': 2}]
_fetch_accepted_list
[{'Error': 'No accepted names were found'}]
_fetch_synonym_data
[{'SynonymName': {'NameId': 13100434, 'ScientificName': 'Quercus pedunculata', 'ScientificNameWithAuthors': 'Quercus pedunculata Ehrh.', 'Family': 'Fagaceae'}, 'AcceptedName': {'N

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,Tropicos,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,robur,L.,Sp. Pl. 2: 996,1753,Accepted,N/A,https://www.tropicos.org/name/13100004,13100004
1,Tropicos,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,pedunculata,Ehrh.,N/A,N/A,Synonym,N/A,https://www.tropicos.org/name/13100434,13100434



  Tropicos  (TropicosAPI)
  Species:   Quercus pedunculata
  Name type: synonym  (synonym of: Quercus robur)
_fetch_query_data
[{'NameId': 13100434, 'ScientificName': 'Quercus pedunculata', 'ScientificNameWithAuthors': 'Quercus pedunculata Ehrh.', 'Family': 'Fagaceae', 'RankAbbreviation': 'sp.', 'NomenclatureStatusID': 3, 'NomenclatureStatusName': 'Invalid', 'Symbol': '**', 'Author': 'Ehrh.', 'DisplayReference': 'Beitr. Naturk. 5: 161', 'DisplayDate': '1790', 'TotalRows': 2}, {'NameId': 50280903, 'ScientificName': 'Quercus pedunculata', 'ScientificNameWithAuthors': 'Quercus pedunculata Hoffm.', 'Family': 'Fagaceae', 'RankAbbreviation': 'sp.', 'NomenclatureStatusName': 'No opinion', 'Author': 'Hoffm.', 'DisplayReference': 'Deutschl. Fl. 1: 338', 'DisplayDate': '1791', 'TotalRows': 2}]
_fetch_accepted_list
[{'SynonymName': {'NameId': 13100434, 'ScientificName': 'Quercus pedunculata', 'ScientificNameWithAuthors': 'Quercus pedunculata Ehrh.', 'Family': 'Fagaceae'}, 'AcceptedName': {'NameI

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,Tropicos,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,robur,L.,Sp. Pl. 2: 996,1753,Accepted,N/A,https://www.tropicos.org/name/13100004,13100004
1,Tropicos,N/A,N/A,N/A,N/A,N/A,N/A,Quercus,pedunculata,Ehrh.,N/A,N/A,Synonym,N/A,https://www.tropicos.org/name/13100434,13100434



  Tropicos  (TropicosAPI)
  Species:   Not species
  Name type: not species
Synonyms returned: 0


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id


---
## FishBase

FishBase is the world's largest online information system for fish species. It does not expose a public API, so this client scrapes two HTML pages per query: the species summary page (which redirects any name — accepted or synonym — to the accepted species page via a `SpecCode`) and the nomenclature page (which lists all names with status, author, and species code as anchor tag parameters). Subspecific names and misspellings are excluded during compilation.

Uses the **standard base-class** `get_synonyms()` pipeline.

In [8]:
from scripts.apis_pipe.fishbase import FishBaseAPI

run_queries([(FISHBASE_PORTAL, FishBaseAPI(), "Gadus morhua", "Gadus callarias")])


  FishBase  (FishBaseAPI)
  Species:   Gadus morhua
  Name type: accepted
_fetch_query_data
    <!-- Display Cookie Consent Banner -->
    <link rel="stylesheet" type="text/css" href="/css/cookie-consent.css">

    <div id="cookie-consent-container" style="display:none;">
        <div id="cookie-consent-banner">
            <div class="cookie-consent-banner-item">
                <p>
                   This website uses cookies to enhance your browsing experience and ensure the functionality of our site. For more detailed information about the types of cookies we use and how we protect your privacy, please visit our <a href='../cookieMoreinfo.php' target='_blank'>Privacy Information</a> page.
                </p>
            </div>
            <div class="cookie-consent-banner-item">
                <button id="accept-all">Accept All</button>
                <button id="accept-necessary">Accept Necessary Cookies Only</button>
                <button id="cookie-settings">Cookie Setting

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Gadus,morhua,"Linnaeus, 1758",N/A,1758,Accepted,N/A,https://www.fishbase.se/nomenclature/69,69
1,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Asellus,major,,N/A,,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
2,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Gadus,callarias,"Linnaeus, 1758",N/A,1758,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
3,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Gadus,vertagus,"Walbaum, 1792",N/A,1792,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
4,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Gadus,heteroglossus,"Walbaum, 1792",N/A,1792,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
5,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Gadus,ruber,"Lacepède, 1803",N/A,1803,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
6,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Gadus,arenosus,"Mitchill, 1815",N/A,1815,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
7,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Gadus,rupestris,"Mitchill, 1815",N/A,1815,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
8,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Morhua,vulgaris,"Fleming, 1828",N/A,1828,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
9,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Morhua,punctatus,"Fleming, 1828",N/A,1828,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69



  FishBase  (FishBaseAPI)
  Species:   Gadus callarias
  Name type: synonym  (synonym of: Gadus morhua)
_fetch_query_data
    <!-- Display Cookie Consent Banner -->
    <link rel="stylesheet" type="text/css" href="/css/cookie-consent.css">

    <div id="cookie-consent-container" style="display:none;">
        <div id="cookie-consent-banner">
            <div class="cookie-consent-banner-item">
                <p>
                   This website uses cookies to enhance your browsing experience and ensure the functionality of our site. For more detailed information about the types of cookies we use and how we protect your privacy, please visit our <a href='../cookieMoreinfo.php' target='_blank'>Privacy Information</a> page.
                </p>
            </div>
            <div class="cookie-consent-banner-item">
                <button id="accept-all">Accept All</button>
                <button id="accept-necessary">Accept Necessary Cookies Only</button>
                <button id="c

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Gadus,morhua,"Linnaeus, 1758",N/A,1758,Accepted,N/A,https://www.fishbase.se/nomenclature/69,69
1,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Asellus,major,,N/A,,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
2,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Gadus,callarias,"Linnaeus, 1758",N/A,1758,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
3,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Gadus,vertagus,"Walbaum, 1792",N/A,1792,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
4,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Gadus,heteroglossus,"Walbaum, 1792",N/A,1792,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
5,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Gadus,ruber,"Lacepède, 1803",N/A,1803,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
6,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Gadus,arenosus,"Mitchill, 1815",N/A,1815,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
7,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Gadus,rupestris,"Mitchill, 1815",N/A,1815,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
8,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Morhua,vulgaris,"Fleming, 1828",N/A,1828,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69
9,FishBase,N/A,N/A,N/A,N/A,N/A,N/A,Morhua,punctatus,"Fleming, 1828",N/A,1828,Synonym,N/A,https://www.fishbase.se/nomenclature/69,69



  FishBase  (FishBaseAPI)
  Species:   Not species
  Name type: not species
_fetch_query_data
    <!-- Display Cookie Consent Banner -->
    <link rel="stylesheet" type="text/css" href="/css/cookie-consent.css">

    <div id="cookie-consent-container" style="display:none;">
        <div id="cookie-consent-banner">
            <div class="cookie-consent-banner-item">
                <p>
                   This website uses cookies to enhance your browsing experience and ensure the functionality of our site. For more detailed information about the types of cookies we use and how we protect your privacy, please visit our <a href='../cookieMoreinfo.php' target='_blank'>Privacy Information</a> page.
                </p>
            </div>
            <div class="cookie-consent-banner-item">
                <button id="accept-all">Accept All</button>
                <button id="accept-necessary">Accept Necessary Cookies Only</button>
                <button id="cookie-settings">Cookie Setti

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id


---
## ITIS (Integrated Taxonomic Information System)

ITIS is an authoritative database of biological names for plants, animals, fungi, and microbes, maintained by a partnership of US federal agencies.

**Custom `get_synonyms()` orchestration.** ITIS exposes data through many small, single-purpose endpoints rather than one compound query. A single lookup requires at least **4 + N** fetch calls, where N is the number of synonyms:

1. `_fetch_query_data` — `getITISTermsFromScientificName` to get the initial TSN
2. `_extract_internal_accepted_id` — calls `_fetch_internal_accepted_id_data` (`getAcceptedNamesFromTSN`) only if the initial result is not already accepted
3. `_fetch_synonym_data` — `getSynonymNamesFromTSN` to get the synonym TSN list, then one `getFullRecordFromTSN` call per synonym
4. `_fetch_accepted_data` — `getFullRecordFromTSN` for the accepted name
5. `_fetch_hierarchy_data` — `getFullHierarchyFromTSN` for the taxonomic hierarchy

Because so many endpoints must be coordinated and each takes the accepted TSN (not raw query data) as input, the orchestration passes the accepted ID explicitly between steps rather than threading raw data through the base-class pipeline. **Expect this section to be noticeably slow.**

In [9]:
from scripts.apis_pipe.itis import ITISAPI

run_queries([(ITIS_PORTAL, ITISAPI(), "Oncorhynchus mykiss", "Salmo mykiss")])


  ITIS  (ITISAPI)
  Species:   Oncorhynchus mykiss
  Name type: accepted
_fetch_query_data
{'author': '(Walbaum, 1792)', 'class': 'gov.usgs.itis.itis_service.data.SvcItisTerm', 'commonNames': ['rainbow trout', 'redband trout', 'truite arc-en-ciel', 'steelhead', 'trucha arcoiris'], 'nameUsage': 'valid', 'scientificName': 'Oncorhynchus mykiss', 'tsn': '161989'}
_fetch_synonym_data
[{'acceptedNameList': {'acceptedNames': [{'acceptedName': 'Oncorhynchus mykiss', 'acceptedTsn': '161989', 'author': None, 'class': 'gov.usgs.itis.itis_service.data.SvcAcceptedName'}], 'class': 'gov.usgs.itis.itis_service.data.SvcAcceptedNameList', 'tsn': '161990'}, 'class': 'gov.usgs.itis.itis_service.data.SvcFullRecord', 'commentList': {'class': 'gov.usgs.itis.itis_service.data.SvcTaxonCommentList', 'comments': [None], 'tsn': '161990'}, 'commonNameList': {'class': 'gov.usgs.itis.itis_service.data.SvcCommonNameList', 'commonNames': [None], 'tsn': '161990'}, 'completenessRating': {'class': 'gov.usgs.itis.itis_s

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,ITIS,Animalia,Chordata,Teleostei,Salmonidae,Salmoniformes,Salmoninae,Oncorhynchus,mykiss,"(Walbaum, 1792)",N/A,,Accepted,"American Fisheries Society Special Publication, no. 12 [1980], American Fisheries Society Special Publication, no. 20 [1991], American Fisheries Society Special Publication, no. 21 [1991], American Fisheries Society Monograph, no. 6 [1992], NODC Taxonomic Code [1996], Fishes of Chesapeake Bay [1997], Federal Register, vol. 63, no. 116 [1998], Federal Register, vol. 63, no. 53 [1998], Federal Register, vol. 64, no. 147 [1999], Federal Register, vol. 65, no. 174 [2000], Federal Register, vol. 66, no. 65 [2001], Bigelow and Schroeder's Fishes of the Gulf of Maine, Third Edition [2002], Catalog of Fishes, 17-Mar-2003 [2003], CDP [2003], American Fisheries Society Special Publication, no. 29 [2004], Federal Register, vol. 71, no. 3 [2006], Federal Register, vol. 71, no. 91 [2007], Federal Register, vol. 74, no. 162 [2009], Federal Register, vol. 76, no. 71 [2011], Federal Register, vol. 78, no. 10 [2013]",https://www.itis.gov/servlet/SingleRpt/SingleRpt?search_topic=TSN&search_value=161989,161989
1,ITIS,N/A,N/A,N/A,N/A,N/A,N/A,Salmo,mykiss,"Walbaum, 1792",N/A,1792,Synonym,"Report of the United States Commissioner of Fisheries for the fiscal year 1928 [1930], NODC Taxonomic Code [1996], Catalog of Fishes, 17-Mar-2003 [2003]",https://www.itis.gov/servlet/SingleRpt/SingleRpt?search_topic=TSN&search_value=161990,161990
2,ITIS,N/A,N/A,N/A,N/A,N/A,N/A,Onchorynchus,mykiss,"(Walbaum, 1792)",N/A,,Synonym,"Environmental Research, vol. 92, no. 3 [2003]",https://www.itis.gov/servlet/SingleRpt/SingleRpt?search_topic=TSN&search_value=633905,633905
3,ITIS,N/A,N/A,N/A,N/A,N/A,N/A,Salmo,gibbsii,"Suckley, 1859",N/A,1859,Synonym,"Catalog of Fishes, 10-Jun-2013 [2013]",https://www.itis.gov/servlet/SingleRpt/SingleRpt?search_topic=TSN&search_value=914074,914074



  ITIS  (ITISAPI)
  Species:   Salmo mykiss
  Name type: synonym  (synonym of: Oncorhynchus mykiss)
_fetch_query_data
{'author': 'Walbaum, 1792', 'class': 'gov.usgs.itis.itis_service.data.SvcItisTerm', 'commonNames': [None], 'nameUsage': 'invalid', 'scientificName': 'Salmo mykiss', 'tsn': '161990'}
_fetch_synonym_data
[{'acceptedNameList': {'acceptedNames': [{'acceptedName': 'Oncorhynchus mykiss', 'acceptedTsn': '161989', 'author': None, 'class': 'gov.usgs.itis.itis_service.data.SvcAcceptedName'}], 'class': 'gov.usgs.itis.itis_service.data.SvcAcceptedNameList', 'tsn': '161990'}, 'class': 'gov.usgs.itis.itis_service.data.SvcFullRecord', 'commentList': {'class': 'gov.usgs.itis.itis_service.data.SvcTaxonCommentList', 'comments': [None], 'tsn': '161990'}, 'commonNameList': {'class': 'gov.usgs.itis.itis_service.data.SvcCommonNameList', 'commonNames': [None], 'tsn': '161990'}, 'completenessRating': {'class': 'gov.usgs.itis.itis_service.data.SvcGlobalSpeciesCompleteness', 'completeness': '',

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,ITIS,Animalia,Chordata,Teleostei,Salmonidae,Salmoniformes,Salmoninae,Oncorhynchus,mykiss,"(Walbaum, 1792)",N/A,,Accepted,"American Fisheries Society Special Publication, no. 12 [1980], American Fisheries Society Special Publication, no. 20 [1991], American Fisheries Society Special Publication, no. 21 [1991], American Fisheries Society Monograph, no. 6 [1992], NODC Taxonomic Code [1996], Fishes of Chesapeake Bay [1997], Federal Register, vol. 63, no. 116 [1998], Federal Register, vol. 63, no. 53 [1998], Federal Register, vol. 64, no. 147 [1999], Federal Register, vol. 65, no. 174 [2000], Federal Register, vol. 66, no. 65 [2001], Bigelow and Schroeder's Fishes of the Gulf of Maine, Third Edition [2002], Catalog of Fishes, 17-Mar-2003 [2003], CDP [2003], American Fisheries Society Special Publication, no. 29 [2004], Federal Register, vol. 71, no. 3 [2006], Federal Register, vol. 71, no. 91 [2007], Federal Register, vol. 74, no. 162 [2009], Federal Register, vol. 76, no. 71 [2011], Federal Register, vol. 78, no. 10 [2013]",https://www.itis.gov/servlet/SingleRpt/SingleRpt?search_topic=TSN&search_value=161989,161989
1,ITIS,N/A,N/A,N/A,N/A,N/A,N/A,Salmo,mykiss,"Walbaum, 1792",N/A,1792,Synonym,"Report of the United States Commissioner of Fisheries for the fiscal year 1928 [1930], NODC Taxonomic Code [1996], Catalog of Fishes, 17-Mar-2003 [2003]",https://www.itis.gov/servlet/SingleRpt/SingleRpt?search_topic=TSN&search_value=161990,161990
2,ITIS,N/A,N/A,N/A,N/A,N/A,N/A,Onchorynchus,mykiss,"(Walbaum, 1792)",N/A,,Synonym,"Environmental Research, vol. 92, no. 3 [2003]",https://www.itis.gov/servlet/SingleRpt/SingleRpt?search_topic=TSN&search_value=633905,633905
3,ITIS,N/A,N/A,N/A,N/A,N/A,N/A,Salmo,gibbsii,"Suckley, 1859",N/A,1859,Synonym,"Catalog of Fishes, 10-Jun-2013 [2013]",https://www.itis.gov/servlet/SingleRpt/SingleRpt?search_topic=TSN&search_value=914074,914074



  ITIS  (ITISAPI)
  Species:   Not species
  Name type: not species
Synonyms returned: 0


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id


---
## GenBank (NCBI Entrez)

GenBank is the NIH genetic sequence database maintained by NCBI. Its Taxonomy division provides accepted names and synonym lists via the Entrez E-utilities (`esearch` + `efetch`); all responses are XML.

Uses the **standard base-class** `get_synonyms()` pipeline.

The call here passes `sleep_between_species=1.0`. NCBI rate-limits unauthenticated callers to **3 requests/second**, and each `get_synonyms()` call makes 2 requests (`esearch` + `efetch`), so sleeping 1 second between species queries keeps well within that limit. Without the sleep, rapid back-to-back calls will be throttled or blocked by NCBI. This is not needed in the normal pipeline when used with the frontend.

In [10]:
from scripts.apis_pipe.genbank import GenBankAPI

run_queries(
    [(GENBANK_PORTAL, GenBankAPI(), "Amanita muscaria", "Agaricus muscarius")],
    sleep_between_species=1.0,
)


  GenBank  (GenBankAPI)
  Species:   Amanita muscaria
  Name type: accepted
_fetch_query_data
{'header': {'type': 'esearch', 'version': '0.3'}, 'esearchresult': {'count': '1', 'retmax': '1', 'retstart': '0', 'idlist': ['41956'], 'translationset': [], 'translationstack': [{'term': 'Amanita muscaria[Scientific Name]', 'field': 'Scientific Name', 'count': '1', 'explode': 'N'}, 'GROUP'], 'querytranslation': 'Amanita muscaria[Scientific Name]'}}
_fetch_synonym_data
<TaxaSet><Taxon>
    <TaxId>41956</TaxId>
    <ScientificName>Amanita muscaria</ScientificName>
    <OtherNames>
        <GenbankCommonName>fly agaric</GenbankCommonName>
        <Synonym>Agaricus muscarius</Synonym>
        <Name>
            <ClassCDE>authority</ClassCDE>
            <DispName>Agaricus muscarius L., 1753</DispName>
        </Name>
        <Name>
            <ClassCDE>authority</ClassCDE>
            <DispName>Amanita muscaria (L.) Lam., 1783</DispName>
        </Name>
        <Name>
            <ClassCDE>missp

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,GenBank,Fungi,Basidiomycota,Agaricomycetes,Amanitaceae,Agaricales,,Amanita,muscaria,(L.) Lam.,N/A,1783,Accepted,N/A,https://www.ncbi.nlm.nih.gov/datasets/taxonomy/41956/,41956
1,GenBank,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,muscarius,L.,N/A,1753,Synonym,N/A,https://www.ncbi.nlm.nih.gov/datasets/taxonomy/41956/,41956



  GenBank  (GenBankAPI)
  Species:   Agaricus muscarius
  Name type: synonym  (synonym of: Amanita muscaria)
_fetch_query_data
{'header': {'type': 'esearch', 'version': '0.3'}, 'esearchresult': {'count': '1', 'retmax': '1', 'retstart': '0', 'idlist': ['41956'], 'translationset': [], 'translationstack': [{'term': 'Agaricus muscarius[All Names]', 'field': 'All Names', 'count': '1', 'explode': 'N'}, 'GROUP'], 'querytranslation': 'Agaricus muscarius[All Names]'}}
_fetch_synonym_data
<TaxaSet><Taxon>
    <TaxId>41956</TaxId>
    <ScientificName>Amanita muscaria</ScientificName>
    <OtherNames>
        <GenbankCommonName>fly agaric</GenbankCommonName>
        <Synonym>Agaricus muscarius</Synonym>
        <Name>
            <ClassCDE>authority</ClassCDE>
            <DispName>Agaricus muscarius L., 1753</DispName>
        </Name>
        <Name>
            <ClassCDE>authority</ClassCDE>
            <DispName>Amanita muscaria (L.) Lam., 1783</DispName>
        </Name>
        <Name>
        

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,GenBank,Fungi,Basidiomycota,Agaricomycetes,Amanitaceae,Agaricales,,Amanita,muscaria,(L.) Lam.,N/A,1783,Accepted,N/A,https://www.ncbi.nlm.nih.gov/datasets/taxonomy/41956/,41956
1,GenBank,N/A,N/A,N/A,N/A,N/A,N/A,Agaricus,muscarius,L.,N/A,1753,Synonym,N/A,https://www.ncbi.nlm.nih.gov/datasets/taxonomy/41956/,41956



  GenBank  (GenBankAPI)
  Species:   Not species
  Name type: not species
Synonyms returned: 0


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id


---
## Symbiota Portals

Symbiota is open-source biodiversity data management software powering numerous natural history collection portals worldwide. Each portal runs its own instance with independent data but the same API structure. `SymbiotaAPI` is instantiated with the portal name, which resolves the correct base URL from `config.py`.

**Custom `get_synonyms()` orchestration.** Unlike the base class, Symbiota resolves the accepted `tid` explicitly before calling the fetch methods. The fetch sequence is:

1. `_fetch_query_data` — performs two calls: a name search to find the initial `tid`, followed by a full `api/v2/taxonomy/{tid}` fetch; returns a combined taxon record
2. `_extract_internal_accepted_id` — reads the accepted `tid` from the combined record (no API call)
3. `_fetch_synonym_data` — scrapes the `/taxa/index.php` HTML page for the accepted taxon's synonym entries
4. `_fetch_accepted_data` — fetches the full `api/v2/taxonomy/{accepted_tid}` JSON record for taxonomy and authorship *(skipped and `raw_data` reused directly if the queried name is already accepted)*

Symbiota portals do not assign separate IDs to synonym records; a single accepted `tid` is used for both `api_internal_id` and `api_link` across every row.

The portals below all share this implementation and are queried in sequence.

In [11]:
from scripts.apis_pipe.symbiota import SymbiotaAPI

_SYMBIOTA_QUERIES = [
    # (portal, accepted_species, synonym_of_accepted)
    (SYMBIOTA_PORTAL_BY_NAME["MyCoPortal"],                       "Agaricus campestris",     "Psalliota villatica"),
    (SYMBIOTA_PORTAL_BY_NAME["Lichen Portal"],                    "Xanthoria parietina",     "Physcia parietina"),
    (SYMBIOTA_PORTAL_BY_NAME["Bryophyte Portal"],                 "Pohlia nutans",           "Webera nutans"),
    (SYMBIOTA_PORTAL_BY_NAME["CCH2"],                             "Heteromeles arbutifolia", "Photinia arbutifolia"),
    (SYMBIOTA_PORTAL_BY_NAME["SERNEC"],                           "Magnolia grandiflora",    "Magnolia foetida"),
    (SYMBIOTA_PORTAL_BY_NAME["NANSH"],                            "Rudbeckia hirta",         "Coreopsis hirta"),
    (SYMBIOTA_PORTAL_BY_NAME["Algae Herbarium Portal"],           "Ulva intestinalis",       "Ilea intestinalis"),
    (SYMBIOTA_PORTAL_BY_NAME["Pterido Portal"],                   "Dryopteris filix-mas",    "Nephrodium filix-mas"),
    (SYMBIOTA_PORTAL_BY_NAME["CNH"],                              "Impatiens capensis",      "Impatiens biflora"),
    (SYMBIOTA_PORTAL_BY_NAME["Mid-Atlantic Herbaria Consortium"], "Quercus rubra",           "Quercus maxima"),
    (SYMBIOTA_PORTAL_BY_NAME["swbiodiversity"],                   "Larrea tridentata",       "Larrea mexicana"),
]

run_queries([
    (portal, SymbiotaAPI(portal.display_name), accepted, synonym)
    for portal, accepted, synonym in _SYMBIOTA_QUERIES
])


  MyCoPortal  (SymbiotaAPI)
  Species:   Agaricus campestris
  Name type: accepted
[MyCoPortal] 'api/v2/taxonomy/search' returned results for 'Agaricus campestris'.
_fetch_query_data
{'tid': 214862, 'kingdomName': 'Fungi', 'rankID': 220, 'rankName': None, 'unitInd1': None, 'unitName1': 'Agaricus', 'unitInd2': None, 'unitName2': 'campestris', 'unitInd3': None, 'unitName3': None, 'cultivarEpithet': None, 'tradeName': None, 'author': 'L.', 'reviewStatus': None, 'displayStatus': None, 'isLegitimate': None, 'source': None, 'notes': None, 'securityStatus': 0, 'modifiedTimestamp': None, 'initialTimestamp': '2011-11-18 23:38:57', 'parentTid': 11736, 'status': 'accepted', 'classification': [{'tid': 7, 'scientificName': 'Fungi', 'author': 'R.T. Moore', 'rankid': 10}, {'tid': 188, 'scientificName': 'Basidiomycota', 'author': 'R.T. Moore', 'rankid': 30}, {'tid': 1128, 'scientificName': 'Agaricomycotina', 'author': 'Doweld', 'rankid': 40}, {'tid': 1431, 'scientificName': 'Agaricomycetes', 'author'

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,MyCoPortal,Fungi,Basidiomycota,Agaricomycetes,Agaricaceae,Agaricales,N/A,Agaricus,campestris,L.,N/A,N/A,Accepted,,https://mycoportal.org/portal/taxa/index.php?tid=214862,214862
1,MyCoPortal,N/A,N/A,N/A,N/A,N/A,N/A,Psalliota,villatica,(Brond.) Bres.],N/A,N/A,Synonym,N/A,https://mycoportal.org/portal/taxa/index.php?taxon=214862,214862



  MyCoPortal  (SymbiotaAPI)
  Species:   Psalliota villatica
  Name type: synonym  (synonym of: Agaricus campestris)
[MyCoPortal] 'api/v2/taxonomy/search' returned results for 'Psalliota villatica'.
_fetch_query_data
{'tid': 527987, 'kingdomName': 'Fungi', 'rankID': 220, 'rankName': None, 'unitInd1': None, 'unitName1': 'Psalliota', 'unitInd2': None, 'unitName2': 'villatica', 'unitInd3': None, 'unitName3': None, 'cultivarEpithet': None, 'tradeName': None, 'author': '(Brond.) Bres.', 'reviewStatus': None, 'displayStatus': None, 'isLegitimate': None, 'source': 'Index Fungorum', 'notes': None, 'securityStatus': 0, 'modifiedTimestamp': None, 'initialTimestamp': '2011-11-19 14:07:42', 'parentTid': 17617, 'status': 'synonym', 'accepted': {'tid': 214862, 'scientificName': 'Agaricus campestris', 'scientificNameAuthorship': 'L.', 'taxonomicSource': None, 'unacceptabilityReason': None, 'taxonRemarks': None}, 'classification': [{'tid': 7, 'scientificName': 'Fungi', 'author': 'R.T. Moore', 'rankid

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,MyCoPortal,Fungi,Basidiomycota,Agaricomycetes,Agaricaceae,Agaricales,N/A,Agaricus,campestris,L.,N/A,N/A,Accepted,,https://mycoportal.org/portal/taxa/index.php?tid=214862,214862
1,MyCoPortal,N/A,N/A,N/A,N/A,N/A,N/A,Psalliota,villatica,(Brond.) Bres.],N/A,N/A,Synonym,N/A,https://mycoportal.org/portal/taxa/index.php?taxon=214862,214862



  MyCoPortal  (SymbiotaAPI)
  Species:   Not species
  Name type: not species
[MyCoPortal] Search endpoint returned no results for 'Not species'. Attempting autocomplete search.
[MyCoPortal] All searches failed or returned no results for 'Not species'.
Synonyms returned: 0


...

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,swbiodiversity,Plantae,Magnoliophyta,,Zygophyllaceae,Zygophyllales,N/A,Larrea,tridentata,(Sessé & Moc. ex DC.) Coville,N/A,N/A,Accepted,,https://swbiodiversity.org/seinet/taxa/index.php?tid=2671,2671
1,swbiodiversity,N/A,N/A,N/A,N/A,N/A,N/A,Larrea,mexicana,Moric.,N/A,N/A,Synonym,N/A,https://swbiodiversity.org/seinet/taxa/index.php?taxon=2671,2671



  swbiodiversity  (SymbiotaAPI)
  Species:   Larrea mexicana
  Name type: synonym  (synonym of: Larrea tridentata)
[swbiodiversity] 'api/v2/taxonomy' returned results for 'Larrea mexicana'.
_fetch_query_data
{'tid': 10527, 'kingdomName': 'Plantae', 'rankID': 220, 'rankName': None, 'unitInd1': None, 'unitName1': 'Larrea', 'unitInd2': None, 'unitName2': 'mexicana', 'unitInd3': None, 'unitName3': None, 'cultivarEpithet': None, 'tradeName': None, 'author': 'Moric.', 'reviewStatus': None, 'displayStatus': None, 'isLegitimate': None, 'source': None, 'sourceIdentifier': None, 'notes': None, 'securityStatus': 0, 'modifiedTimestamp': None, 'initialTimestamp': '1996-11-04 23:57:08', 'parentTid': 4965, 'status': 'synonym', 'accepted': {'tid': 2671, 'scientificName': 'Larrea tridentata', 'scientificNameAuthorship': '(Sessé & Moc. ex DC.) Coville', 'taxonomicSource': None, 'unacceptabilityReason': None, 'taxonRemarks': None}, 'classification': [{'tid': 4965, 'scientificName': 'Larrea', 'author': '

,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
0,swbiodiversity,Plantae,Magnoliophyta,,Zygophyllaceae,Zygophyllales,N/A,Larrea,tridentata,(Sessé & Moc. ex DC.) Coville,N/A,N/A,Accepted,,https://swbiodiversity.org/seinet/taxa/index.php?tid=2671,2671
1,swbiodiversity,N/A,N/A,N/A,N/A,N/A,N/A,Larrea,mexicana,Moric.,N/A,N/A,Synonym,N/A,https://swbiodiversity.org/seinet/taxa/index.php?taxon=2671,2671



  swbiodiversity  (SymbiotaAPI)
  Species:   Not species
  Name type: not species
[swbiodiversity] Search endpoint returned no results for 'Not species'. Attempting autocomplete search.
[swbiodiversity] All searches failed or returned no results for 'Not species'.
Synonyms returned: 0


,api_name,kingdom,phylum,class,family,order,subfamily,genus,species,author,publication_name,publication_year,status,original_source,api_link,api_internal_id
